# StoryForge · Dịch truyện · 10 luồng

1. Chọn **Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU**.
2. Bấm **▶** ở ô **Kết nối Dịch truyện** ngay bên dưới. Bấm **Sao chép mã**,
   dán vào mục **Máy AI cá nhân** trong Dịch truyện của StoryForge và xác nhận.
3. Chờ thông báo **sẵn sàng**, rồi quay về StoryForge để tải truyện và dịch.

<details>
<summary>Thông tin cấu hình và lưu ý · bấm để xem</summary>

**Gemma E2B Q4_K_P · 10 luồng · context 24.000/luồng · output tối đa 12.000 token.**

Model `HauhauCS/Gemma-4-E2B-Uncensored-HauhauCS-Aggressive`, revision
`da8593c3e407afcd3e7da94ff2d69d77e2a28a48`; engine **llama.cpp b12413**.
GPU sampling bật chỉ với đúng model/file/revision này trên GPU; MTP tắt.
Context gồm đầu vào và đầu ra. Không tự giảm cấu hình khi thiếu bộ nhớ.
Muốn đổi model/context/output/số luồng: chờ máy rảnh, đổi biểu mẫu rồi chạy lại ô kết nối.

Cấu hình runtime đã đo trên T4 thật; tốc độ phụ thuộc đầu vào và phiên GPU.
Không cam kết tốc độ hoặc chất lượng bản dịch. Bản giao diện này được kiểm tra
offline và trình duyệt, chưa chạy lại inference GPU sau khi đóng gói.

Không cần Colab Secrets hoặc chép API key. Không chia sẻ mã liên kết hay output
phiên. Giữ xác thực, heartbeat, lease, thu hồi và kiểm tra cấu hình trước khi báo
sẵn sàng. Notebook này chỉ dành cho Dịch truyện; tải TXT và dịch trên StoryForge.

</details>


In [ ]:
#@title Kết nối Dịch truyện · bấm ▶ để bắt đầu
#@markdown Giữ cấu hình mặc định để dùng **E2B Q4 · 10 luồng**. Mã liên kết sẽ hiện bên dưới khi chạy.
thao_tac = "Ghép nối / khởi động" #@param ["Ghép nối / khởi động", "Khôi phục cấu hình trước", "Tạo mã liên kết mới", "Xem trạng thái", "Chẩn đoán", "Dừng máy chủ"]
#@markdown ### Model và thiết bị
model_AI = "Q4_K_P" #@param ["Q4_K_P", "GGUF khác"]
thiet_bi = "Tự động" #@param ["Tự động", "GPU", "CPU"]
so_yeu_cau_dong_thoi = 10 #@param {type:"integer"} {type:"raw"}
#@markdown ### Ngữ cảnh và độ dài trả lời
context_tokens = 24000 #@param {type:"integer"}
max_output_tokens = 12000 #@param {type:"integer"}
#@markdown ### GGUF (khớp sẵn E2B Q4_K_P; chỉ dùng khi chọn GGUF khác)
hf_repo = "HauhauCS/Gemma-4-E2B-Uncensored-HauhauCS-Aggressive" #@param {type:"string"}
gguf_file = "Gemma-4-E2B-Uncensored-HauhauCS-Aggressive-Q4_K_P.gguf" #@param {type:"string"}
hf_revision = "da8593c3e407afcd3e7da94ff2d69d77e2a28a48" #@param {type:"string"}
#@markdown Không cần Colab Secrets. Notebook chỉ nhận credential sau khi người dùng xác nhận mã trong StoryForge.
# File này do build_multiuser_notebook.py sinh; origin không thể đổi trong biểu mẫu.
import base64, io, zipfile, sys, importlib, importlib.metadata, subprocess
SF_ORIGIN = 'https://storyforge-ai-pilot.canhettg113.workers.dev'
if SF_ORIGIN.endswith('.invalid'):
    raise RuntimeError('REVIEW ONLY: origin mẫu .invalid chưa kết nối được tới StoryForge; không gọi mạng.')
try:
    _sf_httpx_version = importlib.metadata.version('httpx')
except importlib.metadata.PackageNotFoundError:
    _sf_httpx_version = None
if _sf_httpx_version != '0.28.1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'httpx==0.28.1'],
                   check=True, timeout=180)
from pathlib import Path
_sf_folder = Path('/content/storyforge-personal-ai/source')
_sf_folder.mkdir(parents=True, exist_ok=True)
_sf_payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_sf_payload))) as _sf_zip:
    _sf_zip.extractall(_sf_folder)
if str(_sf_folder) not in sys.path:
    sys.path.insert(0, str(_sf_folder))
import runtime, connection, settings, linked_connection, start_colab, gateway_translator
for _sf_module in (runtime, connection, settings, linked_connection, start_colab, gateway_translator):
    importlib.reload(_sf_module)
_sf_personal_state = globals().get('SF_STATE', {})
if isinstance(_sf_personal_state, dict) and any(
        _sf_personal_state.get(_sf_name) is not None
        and _sf_personal_state[_sf_name].poll() is None
        for _sf_name in ('model', 'gateway', 'tunnel')):
    raise RuntimeError('Dừng máy chủ notebook cá nhân trong runtime này trước khi ghép nối nhiều người dùng.')
start_colab.ROOT = Path('/content/storyforge-personal-ai')
SF_LINKED_STATE = globals().get('SF_LINKED_STATE', {})
SF_LINKED_STATE['multiuser'] = True
_sf_settings = {'model_choice': model_AI, 'device': thiet_bi,
    'parallel': so_yeu_cau_dong_thoi,
    'context': context_tokens, 'max_tokens': max_output_tokens,
    'repo': hf_repo, 'filename': gguf_file, 'revision': hf_revision}


def _sf_pairing_card(code):
    from html import escape
    return r"""
<style>
.sf-pair {box-sizing:border-box; max-width:720px; padding:24px; margin:12px 0 20px;
  border-radius:16px; background:#3b167f; color:#fff; font:16px/1.5 system-ui,sans-serif;}
.sf-pair h3 {font-size:20px; margin:0 0 16px; color:#fff;}
.sf-pair .sf-row {display:flex; flex-wrap:wrap; align-items:center; gap:16px;}
.sf-pair input {box-sizing:border-box; width:10ch; min-width:0; max-width:100%; padding:8px 0;
  border:0; border-radius:4px; background:transparent; color:#ffe474;
  font:800 clamp(28px,5vw,42px)/1.2 ui-monospace,monospace; letter-spacing:.06em;}
.sf-pair button {min-height:48px; padding:12px 20px; border:0; border-radius:10px;
  background:#7af0de; color:#163c35; font:700 16px/1.5 system-ui,sans-serif; cursor:pointer;}
.sf-pair button:hover {background:#b0fff2;}
.sf-pair button:disabled {cursor:wait; opacity:.7;}
.sf-pair :focus-visible {outline:3px solid #fff; outline-offset:4px;}
.sf-pair input::selection {background:#ffe474; color:#3b167f;}
.sf-pair p {margin:16px 0 0; max-width:65ch; color:#ede2ff;}
.sf-pair [role=status] {margin-top:8px; min-height:24px; color:#fff;}
</style>
<section class="sf-pair" aria-label="Liên kết với StoryForge">
  <h3>Mã liên kết dùng một lần</h3>
  <div class="sf-row">
    <input aria-label="Mã liên kết" value="__SF_PAIR_CODE__" readonly spellcheck="false">
    <button type="button" onclick="(async (button) => {
      const root = button.closest('.sf-pair');
      const field = root.querySelector('input');
      const status = root.querySelector('[role=status]');
      button.disabled = true;
      let copied = false;
      try {
        field.focus(); field.select();
        copied = document.execCommand('copy');
      } catch (_) {}
      if (!copied) {
        try { await navigator.clipboard.writeText(field.value); copied = true; } catch (_) {}
      }
      button.disabled = false;
      if (copied) {
        button.textContent = 'Đã sao chép';
        status.textContent = 'Đã sao chép mã. Quay lại StoryForge để dán.';
        button.focus();
      } else {
        field.focus(); field.select();
        status.textContent = 'Trình duyệt chặn sao chép. Mã đã được chọn: nhấn Ctrl+C (Mac: ⌘C).';
      }
    })(this)">Sao chép mã</button>
  </div>
  <p>Dán mã vào <strong>Máy AI cá nhân</strong> trong Dịch truyện của StoryForge,
  rồi xác nhận đúng tài khoản và máy.</p>
  <div role="status" aria-live="polite">Giữ riêng mã này. Đang chờ xác nhận trong StoryForge.</div>
</section>
""".replace('__SF_PAIR_CODE__', escape(str(code), quote=True))


def _sf_show_pairing_code(code):
    from IPython.display import HTML, display
    display(HTML(_sf_pairing_card(code)))


def _sf_wait_message(status):
    labels = {'pending': 'Đang chờ Anh/Chị xác nhận trong StoryForge…',
              'approved': 'Đã xác nhận. StoryForge đang chuẩn bị kết nối…',
              'provisioning': 'Đang cấp đường kết nối riêng…',
              'slow_down': 'Máy chủ yêu cầu chờ lâu hơn trước lần kiểm tra tiếp theo…'}
    if status in labels:
        print(labels[status], flush=True)

try:
    if thao_tac == 'Ghép nối / khởi động':
        # Check input BEFORE creating/polling a pairing or changing the model.
        _sf_validated = settings.resolve_multiuser_settings(_sf_settings, has_gpu=True)
    if thao_tac == 'Xem trạng thái':
        start_colab.show_linked_panel(SF_LINKED_STATE)
    elif thao_tac == 'Chẩn đoán':
        start_colab.diagnostics(SF_LINKED_STATE)
    elif thao_tac == 'Dừng máy chủ':
        start_colab.stop(SF_LINKED_STATE)
        print('Đã dừng các tiến trình do notebook quản lý. Kết nối sẽ hết lease nếu không còn heartbeat.')
    else:
        if thao_tac == 'Khôi phục cấu hình trước':
            _sf_previous = SF_LINKED_STATE.get('previous_model_config')
            if not _sf_previous:
                raise ValueError('Chưa có cấu hình trước để khôi phục.')
            _sf_settings = {'model_choice': 'GGUF khác', 'device': 'GPU' if _sf_previous['device'] == 'gpu' else 'CPU',
                'parallel': _sf_previous.get('parallel', 1),
                **{key: _sf_previous[key] for key in ('context', 'max_tokens', 'repo', 'filename', 'revision')}}
        _sf_link = linked_connection.LinkedConnection(SF_ORIGIN, SF_LINKED_STATE)
        if thao_tac == 'Tạo mã liên kết mới':
            start_colab.prepare_new_link(SF_LINKED_STATE)
            _sf_pair = _sf_link.new_pair(label='Colab AI')
        else:
            _sf_pair = _sf_link.ensure_pair(label='Colab AI')
        if _sf_pair:
            _sf_show_pairing_code(_sf_pair['code'])
        if SF_LINKED_STATE.get('linked_credentials'):
            _sf_credentials = SF_LINKED_STATE['linked_credentials']
        else:
            print('Đang chờ xác nhận; lần chạy lại sẽ tiếp tục cùng mã liên kết.', flush=True)
            _sf_credentials = _sf_link.poll_until_claim(max_wait=600, on_status=_sf_wait_message)
        print('Đã nhận quyền phiên trong bộ nhớ. Credential không được in ra notebook.', flush=True)
        start_colab.start_linked(SF_LINKED_STATE, _sf_credentials, SF_ORIGIN, settings=_sf_settings)
        start_colab.show_linked_panel(SF_LINKED_STATE)
except KeyboardInterrupt:
    print('Đã dừng lượt khởi động/tải model. Chạy lại ô để thử tiếp; tải lại trang không khởi động lại máy Colab.', flush=True)
    start_colab.show_linked_panel(SF_LINKED_STATE)
except Exception as _sf_error:
    if ((isinstance(_sf_error, (ValueError, TimeoutError)) and type(_sf_error).__name__ != 'JSONDecodeError')
            or isinstance(_sf_error, runtime.RuntimeActionError)):
        print('Chưa áp dụng yêu cầu:', str(_sf_error))
    else:
        print('CHƯA SẴN SÀNG:', type(_sf_error).__name__)
        print('Chi tiết thô và nhật ký riêng tư được ẩn. Chọn Chẩn đoán để xem trạng thái tiến trình.')
    start_colab.show_linked_panel(SF_LINKED_STATE)
